# Đi qua pipeline dữ liệu FD001: từ file text đến bảng cho mô hình

Học viên: Phan Nguyễn Mai Phương — MSHV 25002711

Notebook này chạy lại từng bước xử lý dữ liệu của project trên FD001 và in kết quả trung gian:
dữ liệu được tải về từ đâu, gom các file raw thành một bảng, đưa lên HDFS, lưu thành các lớp Silver/Gold, truy vấn bằng SQL và Spark SQL,
tạo nhãn RUL và đặc trưng cửa sổ trượt, rồi xem quan hệ giữa các bảng. Notebook chỉ đọc dữ liệu,
không ghi gì vào `outputs/`.

In [1]:
from pathlib import Path
import json
import os
import sqlite3
import sys

import numpy as np
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "src" / "cmapss").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 12)

from cmapss.data.contracts import KEY
from cmapss.data.readers import read_observations
from cmapss.data.targets import make_train_targets
from cmapss.etl.pandas_pipeline import build_engine_summary, load_dataset
from cmapss.features.rolling import build_rolling_features
from cmapss.features.contracts import ROLLING_FEATURE_STATS
from cmapss.modeling.contracts import MODELING_FEATURE_COLUMNS
from cmapss.modeling.pipeline import prepare_modeling_dataset

RAW = ROOT / "data" / "raw"

## 1. Dữ liệu gốc

FD001 gồm ba file text không có header. `train` và `test` cùng có 26 cột (id động cơ, cycle, 3 setting,
21 sensor); `RUL_FD001.txt` chỉ có một cột là RUL thật tại cycle cuối của mỗi động cơ test.

In [2]:
rows = []
for name in ["train_FD001.txt", "test_FD001.txt", "RUL_FD001.txt"]:
    lines = (RAW / name).read_text(encoding="utf-8").splitlines()
    rows.append({"file": name, "số dòng": len(lines), "số cột": len(lines[0].split()),
                 "dòng đầu tiên (rút gọn)": " ".join(lines[0].split()[:7]) + " ..."})
pd.DataFrame(rows)

,file,số dòng,số cột,dòng đầu tiên (rút gọn)
0,train_FD001.txt,20631,26,1 1 -0.0007 -0.0004 100.0 518.67 641.82 ...
1,test_FD001.txt,13096,26,1 1 0.0023 0.0003 100.0 518.67 643.02 ...
2,RUL_FD001.txt,100,1,112 ...


### Dữ liệu lấy từ đâu

Các file này không chép tay vào repo mà được tải bằng `python -m cmapss.data.acquire` từ cổng dữ liệu
mở của NASA. Script kiểm tra MD5 của gói ZIP, so SHA-256 của từng file với giá trị đã ghi nhận, và không
bao giờ ghi đè một file raw có nội dung khác. Kết quả lần tải gần nhất nằm trong manifest.

In [3]:
manifest = json.loads((ROOT / "outputs/acquisition/manifest.json").read_text(encoding="utf-8"))
print(manifest["source_url"], "| MD5 gói:", manifest["archive_md5"], "| tải lúc", manifest["retrieved_at_utc"])
pd.DataFrame(manifest["files"]).assign(sha256=lambda d: d["sha256"].str[:12] + "...")

https://data.nasa.gov/docs/legacy/CMAPSSData.zip | MD5 gói: 79a22f36e80606c69d0e9e4da5bb2b7a | tải lúc 2026-10-08T19:52:58+00:00


,name,bytes,sha256,status
0,RUL_FD001.txt,429,a19c8ec94931...,unchanged
1,RUL_FD002.txt,1110,c851dd96a6ea...,unchanged
2,RUL_FD003.txt,428,df1e0566306b...,unchanged
3,RUL_FD004.txt,1084,196b836b85a9...,unchanged
4,test_FD001.txt,2228855,3cda7109ce17...,unchanged
5,test_FD002.txt,5734587,de7b5bf7e998...,unchanged
6,test_FD003.txt,2826651,299babd63c8d...,unchanged
7,test_FD004.txt,6957759,1dc675fff062...,unchanged
8,train_FD001.txt,3515356,963b5e22825b...,unchanged
9,train_FD002.txt,9082480,dac6c4dbc4e7...,unchanged


**Nhận xét.** Cả 12 file của bốn subset đều ở trạng thái `unchanged`: bản tải mới trùng từng byte với bản
đang dùng, nên mọi kết quả trong project được tính trên đúng dữ liệu gốc của NASA. Project chỉ dùng ba file
FD001; ba subset còn lại phục vụ thử nghiệm chuyển miền ở notebook 05.

## 2. Gom train và test thành một bảng

Hai file quan sát có cùng cấu trúc cột nhưng là hai nhóm động cơ khác nhau, nên tôi nối chúng theo
chiều dòng (union) chứ không join. Trước khi nối, mỗi file được gán tên cột, kiểu dữ liệu và thêm cột `split`.
Hàm `read_observations` cũng kiểm tra luôn số cột, giá trị hữu hạn và cycle liên tục.

In [4]:
train = read_observations(RAW / "train_FD001.txt", split="train")
test = read_observations(RAW / "test_FD001.txt", split="test")
print("train:", train.shape, "| test:", test.shape)
train.iloc[:3, :8]

train: (20631, 27) | test: (13096, 27)


,split,engine_id,cycle,setting_1,setting_2,setting_3,sensor_1,sensor_2
0,train,1,1,-0.0007,-0.0004,100.0,518.67,641.82
1,train,1,2,0.0019,-0.0003,100.0,518.67,642.15
2,train,1,3,-0.0043,0.0003,100.0,518.67,642.35


In [5]:
observations = pd.concat([train, test], ignore_index=True)   # bên PySpark là train.unionByName(test)
print("Sau khi nối:", observations.shape)
print("(split, engine_id, cycle) không trùng:", not observations.duplicated(list(KEY)).any())
print("(engine_id, cycle) không trùng:", not observations.duplicated(["engine_id", "cycle"]).any())
observations.query("engine_id == 1 and cycle == 1").iloc[:, :6]

Sau khi nối: (33727, 27)
(split, engine_id, cycle) không trùng: True
(engine_id, cycle) không trùng: False


,split,engine_id,cycle,setting_1,setting_2,setting_3
0,train,1,1,-0.0007,-0.0004,100.0
20631,test,1,1,0.0023,0.0003,100.0


**Nhận xét.** Sau khi nối có 33.727 dòng. Hai dòng ở trên cho thấy vì sao cần cột `split`: động cơ 1 của train
và động cơ 1 của test là hai động cơ khác nhau nhưng trùng `(engine_id, cycle)`. Vì vậy khóa của một quan sát
là `(split, engine_id, cycle)`.

File `RUL_FD001.txt` không được gom vào đây. Nó chỉ được đọc ở bước đánh giá cuối, sau khi đã ghi dự đoán
cho tập test, để tránh rò rỉ nhãn (data leakage).

## 3. Các lớp lưu trữ

Hàm `load_dataset` của bước ETL làm đúng các việc trên (đọc, nối, sắp theo khóa) và tạo luôn nhãn RUL cho train.
Kết quả được lưu theo kiểu Bronze/Silver/Gold: Bronze là file gốc, Silver là dữ liệu đã làm sạch và chuẩn kiểu,
Gold là các bảng đã tổng hợp sẵn để dùng.

In [6]:
silver_obs, silver_targets, _ = load_dataset(RAW, "FD001")
stored_obs = pd.read_parquet("outputs/etl/silver/observations.parquet")
stored_targets = pd.read_parquet("outputs/etl/silver/train_targets.parquet")
engine_summary = pd.read_parquet("outputs/etl/gold/engine_summary.parquet")
stored_features = pd.read_parquet("outputs/features/gold/rolling_features.parquet")

layers = [
    ("Bronze", "data/raw/train_FD001.txt + test_FD001.txt", "1 cycle của 1 động cơ", len(train) + len(test), 26),
    ("Silver", "observations.parquet", "1 cycle của 1 động cơ", *stored_obs.shape),
    ("Silver", "train_targets.parquet", "1 cycle train", *stored_targets.shape),
    ("Gold", "engine_summary.parquet / etl.sqlite", "1 động cơ", len(engine_summary), 5),
    ("Gold", "rolling_features.parquet", "1 window 30 cycles", *stored_features.shape),
]
pd.DataFrame(layers, columns=["lớp", "file", "một dòng là", "số dòng", "số cột"])

,lớp,file,một dòng là,số dòng,số cột
0,Bronze,data/raw/train_FD001.txt + test_FD001.txt,1 cycle của 1 động cơ,33727,26
1,Silver,observations.parquet,1 cycle của 1 động cơ,33727,27
2,Silver,train_targets.parquet,1 cycle train,20631,4
3,Gold,engine_summary.parquet / etl.sqlite,1 động cơ,200,5
4,Gold,rolling_features.parquet,1 window 30 cycles,27927,129


**Nhận xét.** Số dòng không đổi từ Bronze sang Silver (33.727); Silver chỉ thêm cột `split`. Bảng tổng hợp
có 200 dòng, đúng bằng 100 động cơ train cộng 100 động cơ test. Bảng đặc trưng ít dòng hơn bảng quan sát
vì mỗi động cơ cần đủ 30 cycle mới có cửa sổ đầu tiên (xem mục 5).

### 3.1 Lưu trữ phân tán trên HDFS

Ngoài Parquet và SQLite trên đĩa, hai file raw FD001 còn được đưa lên HDFS của một cụm Hadoop 3.4.3 chạy trong
Docker (`infra/hadoop/docker-compose.yml`). Lệnh `python -m cmapss.hdfs` dựng cụm, `put` file lên
`/cmapss/bronze/fd001`, rồi tính lại bảng `engine_summary` bằng hai cách chạy trên HDFS:

- một job MapReduce viết bằng Hadoop Streaming (`src/cmapss/hdfs/lifetime_job.py`), chạy trên YARN;
- một job Spark đọc raw từ HDFS, ghi lớp Silver dạng Parquet lên HDFS rồi đọc lại để tổng hợp.

Notebook không cần Docker: ở đây chỉ đọc kết quả lần chạy đã lưu trong `outputs/hdfs/`. Trước hết xem mapper
và reducer làm gì trên ba dòng đầu của file train.

In [7]:
from cmapss.hdfs import lifetime_job

first_lines = (RAW / "train_FD001.txt").read_text(encoding="utf-8").splitlines()[:3]
mapped = list(lifetime_job.map_lines(first_lines, "/cmapss/bronze/fd001/train_FD001.txt"))
print("map    ->", mapped)
print("reduce ->", list(lifetime_job.reduce_lines(line + "\n" for line in sorted(mapped))))

map    -> ['train:0001\t1', 'train:0001\t2', 'train:0001\t3']
reduce -> ['train\t1\t3\t1\t3']


Mỗi dòng raw thành một cặp khóa `train:0001` và giá trị là cycle. ID động cơ được đệm 0 để khi Hadoop sắp
khóa theo thứ tự chữ, động cơ 2 vẫn đứng trước động cơ 10. Reducer nhận các cycle cùng khóa và in số cycle,
cycle đầu, cycle cuối. Trên toàn bộ dữ liệu, kết quả của cụm như sau.

In [8]:
hdfs_summary = json.loads((ROOT / "outputs/hdfs/summary.json").read_text(encoding="utf-8"))
print(hdfs_summary["hadoop_version"], "| Spark", hdfs_summary["spark_version"])
print((ROOT / "outputs/hdfs/hdfs_listing.txt").read_text(encoding="utf-8"))

mapreduce_table = pd.read_csv(ROOT / "outputs/hdfs/tables/mapreduce_engine_summary.csv")
spark_table = pd.read_csv(ROOT / "outputs/hdfs/tables/spark_engine_summary.csv")
pandas_table = engine_summary.sort_values(["split", "engine_id"]).reset_index(drop=True).astype({"split": str})
pd.DataFrame({
    "so với Pandas": ["MapReduce trên HDFS", "Spark trên HDFS"],
    "số dòng": [len(mapreduce_table), len(spark_table)],
    "giống hệt": [mapreduce_table.equals(pandas_table), spark_table.equals(pandas_table)],
    "thời gian job (giây)": [hdfs_summary["seconds"]["mapreduce_job"], hdfs_summary["seconds"]["spark_job"]],
})

Hadoop 3.4.3 | Spark 4.0.1
drwxr-xr-x   - hadoop supergroup          0 2026-10-08 20:00 /cmapss/bronze
drwxr-xr-x   - hadoop supergroup          0 2026-10-08 20:00 /cmapss/bronze/fd001
-rw-r--r--   1 hadoop supergroup    2228855 2026-10-08 20:00 /cmapss/bronze/fd001/test_FD001.txt
-rw-r--r--   1 hadoop supergroup    3515356 2026-10-08 20:00 /cmapss/bronze/fd001/train_FD001.txt
drwxr-xr-x   - hadoop supergroup          0 2026-10-08 20:00 /cmapss/gold
drwxr-xr-x   - hadoop supergroup          0 2026-10-08 20:00 /cmapss/gold/mapreduce_engine_summary
-rw-r--r--   1 hadoop supergroup          0 2026-10-08 20:00 /cmapss/gold/mapreduce_engine_summary/_SUCCESS
-rw-r--r--   1 hadoop supergroup       3624 2026-10-08 20:00 /cmapss/gold/mapreduce_engine_summary/part-00000
drwxr-xr-x   - root   supergroup          0 2026-10-08 20:00 /cmapss/gold/spark_engine_summary
-rw-r--r--   1 root   supergroup          0 2026-10-08 20:00 /cmapss/gold/spark_engine_summary/_SUCCESS
-rw-r--r--   1 root   supergro

,so với Pandas,số dòng,giống hệt,thời gian job (giây)
0,MapReduce trên HDFS,200,True,22.4
1,Spark trên HDFS,200,True,14.1


**Nhận xét.** Trên HDFS có đủ ba lớp: Bronze là hai file text, Silver là Parquet do Spark ghi (chia thư mục
theo `split=train` và `split=test`), Gold là hai bảng tổng hợp của MapReduce và Spark. Cả hai bảng giống hệt
bảng Pandas ở cả 200 động cơ. Job MapReduce chậm hơn Spark vì mỗi pha ghi kết quả trung gian xuống đĩa và YARN
phải khởi động container cho từng task. Cụm chỉ có một node nên con số thời gian ở đây không nói lên hiệu năng
phân tán; mục đích là chứng minh cùng phép tính chạy đúng trên HDFS.

## 4. Truy vấn dữ liệu

### 4.1 SQL trên SQLite

File SQLite chứa hai bảng nhỏ, `engine_summary` và `train_targets`, có index theo khóa. Truy vấn dưới đây
join hai bảng để xem RUL của động cơ train số 1 tại ba thời điểm.

In [9]:
con = sqlite3.connect("outputs/etl/gold/etl.sqlite")
print(pd.read_sql("SELECT type, name FROM sqlite_master ORDER BY type DESC, name", con))
pd.read_sql("""
    SELECT s.split, s.engine_id, s.n_cycles, s.last_cycle, t.cycle, t.rul
    FROM engine_summary AS s
    JOIN train_targets AS t
      ON t.split = s.split AND t.engine_id = s.engine_id
    WHERE s.split = 'train' AND s.engine_id = 1 AND t.cycle IN (1, 100, 192)
    ORDER BY t.cycle
""", con)

    type                    name
0  table          engine_summary
1  table           train_targets
2  index  idx_engine_summary_key
3  index   idx_train_targets_key


,split,engine_id,n_cycles,last_cycle,cycle,rul
0,train,1,192,192,1,191
1,train,1,192,192,100,92
2,train,1,192,192,192,0


Động cơ 1 sống 192 cycle, nên RUL giảm từ 191 ở cycle 1 xuống 0 ở cycle cuối.

### 4.2 Lọc và chọn cột trên Parquet bằng Pandas

Dữ liệu sensor chi tiết nằm trong Parquet. Ví dụ lấy `sensor_11` của động cơ train 1 trong các cycle 71 đến 100,
tức đúng 30 cycle của cửa sổ kết thúc tại cycle 100.

In [10]:
window_rows = stored_obs.loc[
    stored_obs["split"].eq("train") & stored_obs["engine_id"].eq(1) & stored_obs["cycle"].between(71, 100),
    ["split", "engine_id", "cycle", "sensor_11"],
]
print(len(window_rows), "dòng; trung bình sensor_11 =", round(window_rows["sensor_11"].mean(), 3))
window_rows.tail(3)

30 dòng; trung bình sensor_11 = 47.395


,split,engine_id,cycle,sensor_11
13193,train,1,98,47.37
13194,train,1,99,47.40
13195,train,1,100,47.64


### 4.3 Spark SQL trên cùng file Parquet

Spark chạy ở chế độ local một luồng (`local[1]`) trên máy cá nhân, nên đây chỉ là minh họa cách viết truy vấn,
không phải phép đo hiệu năng. Ô này cần Java; nếu máy không có Java thì ô in thông báo và bỏ qua.

Truy vấn (c) dùng window function để tính trung bình 30 cycle gần nhất, cũng là cách tính đặc trưng
`sensor_11__mean`. Điều kiện lọc cycle phải đặt ở truy vấn ngoài: nếu đặt `WHERE cycle IN (...)` cùng cấp
với window thì window chỉ thấy ba dòng đã lọc và cho kết quả sai.

In [11]:
java_home = ROOT / "tmp" / "java-runtime" / "jdk-17.0.20.1+1-jre"
if "JAVA_HOME" not in os.environ and java_home.exists():
    os.environ["JAVA_HOME"] = str(java_home)
    os.environ["PATH"] = str(java_home / "bin") + os.pathsep + os.environ["PATH"]

try:
    from cmapss.etl.spark_pipeline import create_spark_session
    spark = create_spark_session("data-walkthrough")
except Exception as error:
    spark = None
    print("Bỏ qua Spark SQL:", error)

In [12]:
if spark is not None:
    spark.read.parquet("outputs/etl/silver/observations.parquet").createOrReplaceTempView("observations")
    spark.read.parquet("outputs/etl/silver/train_targets.parquet").createOrReplaceTempView("train_targets")

    print("(a) GROUP BY: số dòng và số động cơ theo split")
    spark.sql("""
        SELECT split, COUNT(*) AS n_rows, COUNT(DISTINCT engine_id) AS n_engines
        FROM observations GROUP BY split ORDER BY split
    """).show()

    print("(b) JOIN theo đủ ba khóa: sensor và nhãn RUL")
    spark.sql("""
        SELECT o.cycle, o.sensor_11, t.rul
        FROM observations o
        JOIN train_targets t
          ON o.split = t.split AND o.engine_id = t.engine_id AND o.cycle = t.cycle
        WHERE o.split = 'train' AND o.engine_id = 1 AND o.cycle BETWEEN 98 AND 100
        ORDER BY o.cycle
    """).show()

    print("(c) Window function: trung bình 30 cycles gần nhất của sensor_11")
    spark.sql("""
        SELECT engine_id, cycle, n_rows_in_window, ROUND(sensor_11_mean30, 3) AS sensor_11_mean30
        FROM (
            SELECT engine_id, cycle,
                   COUNT(*) OVER w AS n_rows_in_window,
                   AVG(sensor_11) OVER w AS sensor_11_mean30
            FROM observations
            WHERE split = 'train' AND engine_id = 1
            WINDOW w AS (PARTITION BY split, engine_id ORDER BY cycle
                         ROWS BETWEEN 29 PRECEDING AND CURRENT ROW)
        )
        WHERE cycle IN (29, 30, 100)
        ORDER BY cycle
    """).show()
    spark.stop()

(a) GROUP BY: số dòng và số động cơ theo split


+-----+------+---------+
|split|n_rows|n_engines|
+-----+------+---------+
| test| 13096|      100|
|train| 20631|      100|
+-----+------+---------+

(b) JOIN theo đủ ba khóa: sensor và nhãn RUL


+-----+---------+---+
|cycle|sensor_11|rul|
+-----+---------+---+
|   98|    47.37| 94|
|   99|     47.4| 93|
|  100|    47.64| 92|
+-----+---------+---+

(c) Window function: trung bình 30 cycles gần nhất của sensor_11


+---------+-----+----------------+----------------+
|engine_id|cycle|n_rows_in_window|sensor_11_mean30|
+---------+-----+----------------+----------------+
|        1|   29|              29|          47.279|
|        1|   30|              30|          47.283|
|        1|  100|              30|          47.395|
+---------+-----+----------------+----------------+



**Nhận xét.** Ở truy vấn (c), tại cycle 29 cửa sổ mới có 29 dòng nên pipeline không tạo đặc trưng cho cycle này;
từ cycle 30 cửa sổ mới đủ 30 dòng. Giá trị tại cycle 100 là 47,395, khớp với kết quả Pandas ở mục 4.2.

## 5. Các bước biến đổi

Biến đổi ở đây là chữ T trong ETL: đổi kiểu, nối, tổng hợp, tạo nhãn và tạo đặc trưng.

### 5.1 Tạo nhãn RUL

Với mỗi động cơ train, RUL = cycle lớn nhất của động cơ đó − cycle hiện tại.

In [13]:
targets = make_train_targets(train)
targets.query("engine_id == 1 and cycle in [1, 100, 192]")

,split,engine_id,cycle,rul
0,train,1,1,191
99,train,1,100,92
191,train,1,192,0


### 5.2 Tổng hợp theo động cơ (bảng Gold `engine_summary`)

Tương đương `GROUP BY split, engine_id`: mỗi động cơ còn một dòng với số cycle, cycle đầu và cycle cuối.

In [14]:
summary = build_engine_summary(observations)
print(summary.shape)
summary.query("engine_id == 1")

(200, 5)


,split,engine_id,n_cycles,first_cycle,last_cycle
0,test,1,31,1,31
100,train,1,192,1,192


### 5.3 Đặc trưng cửa sổ trượt 30 cycle

Với mỗi động cơ, một cửa sổ (trailing window) gồm 30 cycle gần nhất trượt dần theo thời gian. Trong mỗi cửa sổ,
mỗi sensor cho sáu thống kê: giá trị cuối, trung bình, độ lệch chuẩn, min, max và độ dốc (slope) của đường
hồi quy tuyến tính theo thời gian. Để chắc mình hiểu đúng hàm `build_rolling_features`, tôi tính tay sáu thống kê
của `sensor_11` tại cycle 100 của động cơ train 1 rồi so với kết quả của hàm.

In [15]:
z = window_rows["sensor_11"].to_numpy()        # 30 giá trị, cycle 71 đến 100
k = np.arange(1, len(z) + 1)
by_hand = {
    "last": z[-1], "mean": z.mean(), "std": z.std(ddof=1), "min": z.min(), "max": z.max(),
    "slope": ((k - k.mean()) * (z - z.mean())).sum() / ((k - k.mean()) ** 2).sum(),
}

features = build_rolling_features(silver_obs, window_cycles=30)
row = features.query("split == 'train' and engine_id == 1 and cycle == 100").iloc[0]
pd.DataFrame({
    "tính tay": pd.Series(by_hand),
    "build_rolling_features": pd.Series({s: row[f"sensor_11__{s}"] for s in ROLLING_FEATURE_STATS}),
}).round(6)

,tính tay,build_rolling_features
last,47.640000,47.640000
mean,47.395000,47.395000
std,0.105332,0.105332
min,47.220000,47.220000
max,47.640000,47.640000
slope,0.003669,0.003669


Hai cột trùng nhau. Tiếp theo kiểm tra bảng đặc trưng vừa tính lại có giống file Parquet đã lưu không,
và số dòng có đúng bằng tổng (số cycle − 29) của các động cơ không.

In [16]:
pd.testing.assert_frame_equal(features, stored_features, check_dtype=False)
n_cycles = observations.groupby(["split", "engine_id"]).size()
print("Số dòng đặc trưng:", len(features), "| tổng (số cycle - 29):", int((n_cycles - 29).sum()))
print(features["split"].value_counts().to_dict())

Số dòng đặc trưng: 27927 | tổng (số cycle - 29): 27927
{'train': 17731, 'test': 10196}


## 6. Những cột nào vào mô hình

Mỗi sensor sinh ra sáu cột tên dạng `<sensor>__<thống kê>`, tổng cộng 21 × 6 = 126 cột.

In [17]:
roles = pd.DataFrame([
    ("split, engine_id, cycle", 3, "khóa: join, chia fold theo động cơ", "không"),
    ("setting_1..setting_3", 3, "điều kiện vận hành; FD001 gần như không đổi", "không"),
    ("sensor_1..sensor_21", 21, "tín hiệu gốc; chỉ dùng qua đặc trưng window", "không (trực tiếp)"),
    ("sensor_k__{last, mean, std, min, max, slope}", 126, "đặc trưng window 30 cycles", "có"),
    ("rul", 1, "nhãn (biến mục tiêu y)", "là y"),
], columns=["nhóm cột", "số cột", "vai trò", "trong X?"])
roles

,nhóm cột,số cột,vai trò,trong X?
0,"split, engine_id, cycle",3,"khóa: join, chia fold theo động cơ",không
1,setting_1..setting_3,3,điều kiện vận hành; FD001 gần như không đổi,không
2,sensor_1..sensor_21,21,tín hiệu gốc; chỉ dùng qua đặc trưng window,không (trực tiếp)
3,"sensor_k__{last, mean, std, min, max, slope}",126,đặc trưng window 30 cycles,có
4,rul,1,nhãn (biến mục tiêu y),là y


In [18]:
dataset = prepare_modeling_dataset(stored_features, stored_targets)   # join theo 3 khóa
X = dataset.loc[:, list(MODELING_FEATURE_COLUMNS)]
y = dataset["rul"]
print("X:", X.shape, "| y:", y.shape)
print("Vài tên cột của X:", list(X.columns[:6]), "...")

X: (17731, 126) | y: (17731,)
Vài tên cột của X: ['sensor_1__last', 'sensor_1__mean', 'sensor_1__std', 'sensor_1__min', 'sensor_1__max', 'sensor_1__slope'] ...


**Nhận xét.** Mô hình nhận đủ 126 cột, không có bước chọn đặc trưng riêng. Các sensor hằng vẫn nằm trong X nhưng
không mang thông tin; mô hình cây sẽ không dùng chúng để chia nhánh. X có 17.731 dòng, chính là số dòng đặc trưng
của tập train.

## 7. Quan hệ giữa các bảng

Tôi kiểm tra từng quan hệ trực tiếp trên dữ liệu bằng cách đếm số dòng khớp sau khi join.

In [19]:
per_engine = observations.groupby(["split", "engine_id"]).size()
checks = [("engine_summary -- observations", "1 : N",
           f"mỗi động cơ có {per_engine.min()}--{per_engine.max()} cycles")]

obs_train = observations.query("split == 'train'")[list(KEY)]
m = obs_train.merge(stored_targets, on=list(KEY), how="outer", indicator=True)
checks.append(("observation train -- train_target", "1 : 1",
               f"{(m['_merge'] == 'both').sum()} / {len(obs_train)} dòng khớp"))

m = observations[list(KEY)].merge(stored_features[list(KEY)], on=list(KEY), how="left", indicator=True)
checks.append(("observation -- dòng đặc trưng", "1 : 0..1",
               f"{(m['_merge'] == 'both').sum()} có đặc trưng, {(m['_merge'] == 'left_only').sum()} chưa đủ window"))

n_train_features = int(stored_features["split"].eq("train").sum())
checks.append(("đặc trưng train -- train_target", "1 : 1",
               f"{len(dataset)} dòng sau join = {n_train_features} dòng đặc trưng train"))

final_eval = pd.read_parquet("outputs/modeling/predictions/final_evaluation.parquet")
checks.append(("dự đoán test -- nhãn test", "1 : 1",
               f"{len(final_eval)} dòng, {final_eval['engine_id'].nunique()} động cơ test khác nhau"))
pd.DataFrame(checks, columns=["hai bảng", "quan hệ", "kiểm tra trên dữ liệu"])

,hai bảng,quan hệ,kiểm tra trên dữ liệu
0,engine_summary -- observations,1 : N,mỗi động cơ có 31--362 cycles
1,observation train -- train_target,1 : 1,20631 / 20631 dòng khớp
2,observation -- dòng đặc trưng,1 : 0..1,"27927 có đặc trưng, 5800 chưa đủ window"
3,đặc trưng train -- train_target,1 : 1,17731 dòng sau join = 17731 dòng đặc trưng train
4,dự đoán test -- nhãn test,1 : 1,"100 dòng, 100 động cơ test khác nhau"


**Nhận xét.** Chỉ có một quan hệ một-nhiều là động cơ với các quan sát của nó (31 đến 362 cycle mỗi động cơ).
Các join còn lại là một-một, hoặc một với không/một như quan sát với dòng đặc trưng: 5.800 quan sát đầu đời
chưa đủ cửa sổ 30 cycle nên không có đặc trưng. Không có quan hệ nhiều-nhiều. Hai cửa sổ liên tiếp dùng chung
29 quan sát, nhưng đó là chồng lấn khi tính toán theo thời gian, không phải một phép join nhiều-nhiều.

Ô cuối cho thấy chuyện gì xảy ra nếu join sai khóa, chỉ theo `engine_id`.

In [20]:
f1 = stored_features.query("split == 'train' and engine_id == 1")
t1 = stored_targets.query("engine_id == 1")
wrong = f1.merge(t1, on=["engine_id"])
right = f1.merge(t1, on=list(KEY))
print(f"đặc trưng: {len(f1)} dòng, nhãn: {len(t1)} dòng")
print(f"join chỉ theo engine_id : {len(wrong):>6} dòng  (= {len(f1)} x {len(t1)}, sai)")
print(f"join theo đủ 3 khóa     : {len(right):>6} dòng  (đúng)")

đặc trưng: 163 dòng, nhãn: 192 dòng
join chỉ theo engine_id :  31296 dòng  (= 163 x 192, sai)
join theo đủ 3 khóa     :    163 dòng  (đúng)


## Tóm tắt

Dữ liệu được tải từ NASA bằng script có kiểm tra MD5 và SHA-256. Train và test được đọc riêng, thêm cột `split` rồi nối theo dòng; file nhãn test để riêng đến bước đánh giá cuối.
Dữ liệu lưu ở Parquet (Silver, Gold), riêng bảng tổng hợp và nhãn có thêm bản SQLite để truy vấn bằng SQL;
Spark SQL và Pandas đọc thẳng Parquet. Bản raw và lớp Silver còn được lưu trên HDFS, nơi MapReduce và Spark
tính lại bảng Gold và cho kết quả giống hệt Pandas. Thứ tự biến đổi là: đổi kiểu, nối, tạo nhãn RUL, tổng hợp theo động cơ,
tạo đặc trưng cửa sổ 30 cycle, rồi join với nhãn theo đủ ba khóa. Mô hình chỉ nhận 126 cột đặc trưng.
Các bảng có quan hệ một-nhiều (động cơ với quan sát) hoặc một-một, không có nhiều-nhiều.